# Start-to-Finish Cartesian Wave Project

Generate, build, run, and inspect the standalone Cartesian wave project from start to finish.

Navigation: [Index](../index.ipynb) | Previous: [Wave Equation and C Code Generation](wave_equation_and_c_codegen.ipynb) | Next: [Reference-Metric Applications](../4-curvilinear/reference_metric_applications.ipynb)


## Generate the Real Project
The command writes the same project inspected in the Cartesian wave project lesson, now as the complete Cartesian wave workflow.

## Import Cartesian Project Execution Helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.


In [ ]:
from pathlib import Path
import re, shutil, subprocess, sys, tempfile


def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")


def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(
            args,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)


def require_toolchain():
    if shutil.which("make") is None:
        raise RuntimeError(
            "This notebook requires make to build the generated project."
        )
    if not any(shutil.which(name) for name in ["cc", "gcc", "clang"]):
        raise RuntimeError(
            "This notebook requires a C compiler such as cc, gcc, or clang."
        )


## Create a Cartesian Project Workspace

The workspace keeps generated files separate from the tutorial source tree.


In [ ]:
PROJECT_NAME = "wave_equation_cartesian"
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_cartesian_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME


## Generate the Cartesian Wave Project

This command invokes the same module a learner can run from a terminal and then verifies that the project directory exists.


In [ ]:
command = [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"]
print("generator command: python -m nrpy.examples.wave_equation_cartesian")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)
print("project path: project/wave_equation_cartesian")


## Inspect Files and Build

## Step 4: Confirm the Generated Project Directory

Only after generation do we name the generated project files. The inventory is intentionally compact.

In [ ]:
required = [
    "Makefile",
    "BHaH_function_prototypes.h",
    "rhs_eval.c",
    "wave_equation_cartesian.par",
]
for relative_path in required:
    path = PROJECT_DIR / relative_path
    if not path.is_file():
        raise FileNotFoundError(path)
print("selected generated files:")
for relative_path in required:
    print(relative_path)
for subdir in ["MoL", "diagnostics"]:
    count = (
        sum(1 for path in (PROJECT_DIR / subdir).glob("*.c"))
        if (PROJECT_DIR / subdir).is_dir()
        else 0
    )
    print(f"{subdir}/ C files:", count)


## Step 5: Inspect the Parameter File

The parameter file is the runtime interface for the generated executable.

In [ ]:
print(
    (PROJECT_DIR / "wave_equation_cartesian.par").read_text(
        encoding="utf-8", errors="replace"
    )
)


## Step 6: Inspect Build Rules

Only the leading Makefile lines are needed here; the anatomy notebook returns to file structure later.

In [ ]:
for line in (
    (PROJECT_DIR / "Makefile")
    .read_text(encoding="utf-8", errors="replace")
    .splitlines()[:35]
):
    print(line)


## Step 7: Inspect Function Prototypes

The prototype header lists callable generated functions.

In [ ]:
for line in (
    (PROJECT_DIR / "BHaH_function_prototypes.h")
    .read_text(encoding="utf-8", errors="replace")
    .splitlines()
):
    if "rhs_eval" in line or "diagnostics" in line or line.startswith("void "):
        print(line)


## Step 8: Inspect the RHS Kernel Excerpt

This excerpt shows the generated update kernel without turning the first project lesson into file anatomy.

In [ ]:
lines = (
    (PROJECT_DIR / "rhs_eval.c")
    .read_text(encoding="utf-8", errors="replace")
    .splitlines()
)
for line in lines[:60]:
    print(line)


## Run and Interpret Diagnostics
The run is shortened for notebook execution, but it still exercises the generated executable and diagnostic writer.

## Step 9: Shorten runtime parameters

Only runtime values are changed so the notebook run finishes quickly.

In [ ]:
parfile = PROJECT_DIR / "wave_equation_cartesian.par"
par_text = parfile.read_text(encoding="utf-8")
par_text = par_text.replace("t_final = 8.0", "t_final = 0.2")
par_text = par_text.replace(
    "diagnostics_output_every = 0.2", "diagnostics_output_every = 0.1"
)
par_text = par_text.replace(
    "output_progress_every = 1", "output_progress_every = 1000000"
)
parfile.write_text(par_text, encoding="utf-8")
print(f"--- runtime {parfile.name} ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))


## Step 10: Build the executable

The build step compiles generated C after checking that external build tools are available.

In [ ]:
require_toolchain()
build_output = run_command(["make", "-j2"], PROJECT_DIR, timeout=300)
print("build completed")
print("compiler output line count:", len(build_output.splitlines()))


## Step 11: Run the Default Resolution

The default run writes the first diagnostic file.

In [ ]:
default_output = run_command([f"./{PROJECT_NAME}"], PROJECT_DIR, timeout=90)
print("run output (default resolution):")
for line in default_output.splitlines()[:12]:
    if line.strip():
        print(line.rstrip())


## Step 12: Run a Refined Resolution

The convergence-factor run repeats the same executable at higher resolution.

In [ ]:
refined_output = run_command([f"./{PROJECT_NAME}", "2.0"], PROJECT_DIR, timeout=90)
print("run output (convergence factor 2.0):")
for line in refined_output.splitlines()[:12]:
    if line.strip():
        print(line.rstrip())


## Compare Cartesian Diagnostic Errors

The output is used directly in the next step.


In [ ]:
diagnostic_rows = {}
for diagnostic in sorted(PROJECT_DIR.glob("out0d-conv_factor*.txt")):
    rows = [
        [float(value) for value in line.split()]
        for line in diagnostic.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()
        if line.strip()
    ]
    if len(rows) < 2:
        raise RuntimeError(f"Expected at least two data rows in {diagnostic.name}.")
    diagnostic_rows[diagnostic.name] = rows
    print(diagnostic.name, "rows:", len(rows), "last row:", rows[-1])
coarse = diagnostic_rows["out0d-conv_factor1.00.txt"][-1][1]
fine = diagnostic_rows["out0d-conv_factor2.00.txt"][-1][1]
print("last-row uu error ratio coarse/fine:", coarse / fine)
if fine >= coarse:
    raise RuntimeError("Expected the refined run to reduce the uu error.")


The runtime parameter file matches the displayed diagnostics. The two diagnostic files show the same Cartesian wave project running at two resolutions after generation and compilation.


## Continue to Curvilinear Coordinates
- [Boundary Conditions and Convergence](../2-numerical_methods/boundary_conditions_and_convergence.ipynb)
- [Reference-Metric Applications](../4-curvilinear/reference_metric_applications.ipynb)
- [BHaH Project Anatomy](../5-infrastructures/bhah_project_anatomy.ipynb)
